In [1]:
!pip install --upgrade mlflow scikit-learn pandas matplotlib

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)


In [2]:
# Import all required libraries
import mlflow
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
mlflow.set_experiment("iris-demo")

print("Experiment 'iris-demo' is set")

2026/06/28 05:04:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/28 05:04:32 INFO mlflow.store.db.utils: Updating database tables
2026/06/28 05:04:38 INFO mlflow.tracking.fluent: Experiment with name 'iris-demo' does not exist. Creating a new experiment.


Experiment 'iris-demo' is set


In [4]:
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 112
Testing samples: 38


In [5]:
params = {
    "solver": "lbfgs",
    "max_iter": 100,
    "random_state": 42
}

with mlflow.start_run(run_name="baseline"):

    # Log parameters
    mlflow.log_params(params)

    # Train model
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)

    # Evaluate
    acc = accuracy_score(y_test, preds)

    # Log metric
    mlflow.log_metric("accuracy", acc)

    # Save model and register it
    mlflow.sklearn.log_model(model, artifact_path="model", registered_model_name="Iris_Classifier")

    print("Baseline Accuracy:", acc)
    print("Run ID:", mlflow.active_run().info.run_id)

2026/06/28 05:06:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/28 05:06:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'Iris_Classifier'.


Baseline Accuracy: 1.0
Run ID: ad2b46a2a1c94b63b83c1eb72f2d2022


Created version '1' of model 'Iris_Classifier'.


In [6]:
params = {
    "solver": "lbfgs",
    "max_iter": 100,
    "C": 0.5,
    "random_state": 42
}

with mlflow.start_run(run_name="C=0.5"):

    mlflow.log_params(params)

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_metric("accuracy", acc)

    print("Accuracy with C=0.5:", acc)

Accuracy with C=0.5: 1.0


In [7]:
import mlflow

client = mlflow.tracking.MlflowClient()
registered_models = client.search_registered_models()

if registered_models:
    # Get the name of the first registered model
    correct_model_name = registered_models[0].name
    model_uri = f"models:/{correct_model_name}/latest"
    print(f"Attempting to load model: {model_uri}")
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    print("Model loaded successfully!")
else:
    print("No registered models found. Please register a model first, or update `model_uri` with the correct model name if it's already registered.")

Attempting to load model: models:/Iris_Classifier/latest
Model loaded successfully!


In [9]:
!pip install mlflow pyngrok

In [10]:
import os

os.system("mlflow ui --host 0.0.0.0 --port 5000 &")

0

In [12]:
from pyngrok import ngrok

ngrok.set_auth_token("3FkhBNcMZqmwQiyVy9A9aD6kmoj_54Wp7GSkjQt8R6f8N1AiH")

In [13]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://decimeter-lusty-barbell.ngrok-free.dev" -> "http://localhost:5000"
